# ColabUMA Conf — conformer search with UMA

Searches the **conformers** of a system such as a metal complex with a bound substrate, using Meta's machine learning potential **UMA**, and reports the few lowest in energy (5 by default). UMA's training data (OMol25) holds many metal complexes, so the surroundings of a metal, where force fields and semiempirical methods struggle, need no special treatment. No coding needed.

1. Optimize the input structure and read its **bonding** (which atoms are bonded to which)
2. Run several **high-temperature molecular dynamics (MD)** runs and collect structures along them (snapshots)
3. Optimize the snapshots quickly, then optimize only the lowest ones fully
4. Drop structures whose bonding changed and duplicates of the same conformer, and sort by energy

## How to use

1. In **Runtime → Change runtime type**, choose **T4 GPU**
2. First time only: open 🔑 (Secrets) in the left sidebar, add your Hugging Face access token with the name `HF_TOKEN`, and turn on notebook access
3. In **1. Input form** below, set the output name, charge, spin multiplicity and search settings
4. Choose **Runtime → Run all**
5. A file selection button appears in **2. File selection and checks**. Choose one xyz or Gaussian gjf / com file holding **the metal complex with the substrate placed on it**. You can leave it running after that
6. Check the conformer table, 3D view and energy plot in the cells below, and download the ZIP of results in the last cell

Only the structure is read from a gjf; the charge and multiplicity come from the form (the values of the whole system, complex and substrate together). To run again, repeat steps 3–6 (the install and the model load are skipped). In the file selection, Cancel upload reuses the previous file.

> **Before you start**: the model weights, [facebook/UMA](https://huggingface.co/facebook/UMA), require an access request. Log in to Hugging Face, accept the FAIR Chemistry License v1 on that page, and once approved, create a token with Read access under [Access Tokens](https://huggingface.co/settings/tokens).

## You do not say how the substrate binds

There is no setting for whether the substrate is coordinated to the metal or only held by hydrogen bonds or π interactions. The pairs of atoms that are bonded in the optimized input structure (metal–donor pairs included) are kept as they are.

- The substrate is **coordinated to the metal** in the input → the coordination is kept and only the conformation changes
- The substrate is **only attached** (not bonded) in the input → its position and orientation change too. A spherical wall around the whole system keeps it from drifting away

During the MD, a restraint pulls a bond back only when it stretches too far. After the optimizations, structures with a broken bond, a new bond or an inverted stereocentre are discarded. With `keep_coordination` off, pairs that involve a metal are not watched, so structures where a ligand or the substrate has bound to or left the metal are kept as well.

## Search settings

| Item | Default | Description |
|---|---|---|
| `n_conformers` | 5 | Number of conformers to report |
| `md_temperature` | 600 | MD temperature (K). Hotter moves further but breaks more structures |
| `md_time_ps` | 20 | MD time (ps, all MD runs together). The time step is 2 fs (hydrogen has a mass of 4 during the MD only) |
| `n_replicas` | 4 | Number of MD runs, started from the same structure with different velocities |
| `n_snapshots` | 100 | Number of structures taken from the MD at even intervals |
| `n_refine` | 20 | Number of candidates optimized fully after the quick optimization |
| `keep_coordination` | on | Keep the bonds around the metal as in the input structure |
| Convergence | normal (0.02) | fmax (eV/Å) of the full optimization |

The run time is set almost entirely by the number of force evaluations: with the defaults, 10,000 for the MD and a few thousand to 10,000 for the optimizations together. The progress bars show an estimate of the time left. If it takes too long, lower `md_time_ps`, `n_snapshots` or `n_refine`.

## Output (contents of the ZIP)

| File | Contents |
|---|---|
| `<name>_conformers.xyz` | The reported conformers (lowest energy first, multi-structure xyz), superimposed on the first |
| `<name>_conf1.xyz` … | Each conformer on its own |
| `<name>_conformers.csv` | ΔE (kcal/mol), population at 298 K, energies, RMSD to the first, convergence |
| `<name>_ensemble.xyz` | Every conformer left after the full optimization (those past the reported number too) |
| `<name>_input_opt.xyz` | The optimized input structure |
| `<name>_dft.csv` | DFT single point energies (when the DFT single point is on) |
| `<name>_log.txt` | Input settings, software versions, the course of the search, the energy of each conformer |

## Notes

- A search by high-temperature MD is not guaranteed to find every conformer. For large, flexible molecules, make `md_time_ps` longer or change `random_seed` and check that the same lowest structure comes out.
- The results are conformers **with the same bonding as the input structure**. For isomers that coordinate differently (another site, cis / trans and so on), run a search from each and compare the energies.
- The populations are a rough guide from ΔE (electronic energy) among the reported conformers only. Vibrational and solvent contributions are not included.
- The charge and spin multiplicity are the same for all structures. If another spin state is possible, run it separately with that multiplicity.
- UMA is a machine learning potential, not a replacement for DFT. Check final energy comparisons with DFT (`run_dft` computes single points of the reported conformers).
- For important structures, run the vibrational analysis in [ColabUMA_Opt](https://colab.research.google.com/github/RChibaChiba/ColabChem/blob/main/ColabUMA_Opt_en.ipynb) to confirm they are minima.
- Periodic systems (crystals, surfaces) are not supported.

## License

- This notebook and the shared code (`colabuma`): [MIT License](https://github.com/RChibaChiba/ColabChem/blob/main/LICENSE)
- UMA model weights: [FAIR Chemistry License v1](https://huggingface.co/facebook/UMA)
- Libraries installed at runtime are used under their own licenses. See [License in the README](https://github.com/RChibaChiba/ColabChem/blob/main/README.md#license)

## Citation

If you use this notebook in a publication, please cite:

- B. M. Wood *et al.*, "UMA: A Family of Universal Models for Atoms", arXiv:2506.23971 (2025).
- D. S. Levine *et al.*, "The Open Molecules 2025 (OMol25) Dataset, Evaluations, and Models", arXiv:2505.08762 (2025).
- fairchem: https://github.com/facebookresearch/fairchem

If you use the DFT single point, please also cite:

- PySCF (Q. Sun *et al.*, *J. Chem. Phys.* **153**, 024109 (2020))
- GPU4PySCF (X. Wu *et al.*, arXiv:2404.09452 (2024))

In [ ]:
#@title 1. Input form { display-mode: "form" }
#@markdown When the form is filled in, choose **Runtime → Run all**. Pick an xyz or gjf file in the next cell; after that you can leave it running.

#@markdown ### Output
output_name = "result"  #@param {type:"string"}

#@markdown ### Structure
#@markdown Charge and spin multiplicity of the whole system (metal complex and substrate together).
charge = 0  #@param {type:"integer"}
multiplicity = 1  #@param {type:"integer"}

#@markdown ### Conformer search
#@markdown Number of conformers to report (lowest energy first).
n_conformers = 5  #@param {type:"integer"}
#@markdown Temperature (K) and total time (ps, all replicas together) of the MD that explores the conformers. Hotter and longer finds more, but takes longer.
md_temperature = 600  #@param {type:"number"}
md_time_ps = 20  #@param {type:"number"}
#@markdown Number of independent MD runs (the total time is split between them) and of snapshots taken from them.
n_replicas = 4  #@param {type:"integer"}
n_snapshots = 100  #@param {type:"integer"}
#@markdown After a quick optimization of all snapshots, this many of the lowest are optimized fully.
n_refine = 20  #@param {type:"integer"}
#@markdown Keep the metal–ligand contacts of the input structure. Turn off to let ligands or the substrate bind to and leave the metal.
keep_coordination = True  #@param {type:"boolean"}
random_seed = 0  #@param {type:"integer"}

#@markdown ### Settings
model_name = "uma-s-1p1"  #@param ["uma-s-1p1", "uma-s-1p2", "uma-m-1p1"]
task_name = "omol"  #@param ["omol"]
optimizer_name = "LBFGS"  #@param ["LBFGS", "BFGS", "FIRE"]
#@markdown Convergence threshold fmax (eV/Å) of the full optimization. normal is close to Gaussian's default, tight to Gaussian's opt=tight.
convergence = "normal (0.02)"  #@param ["loose (0.05)", "normal (0.02)", "tight (0.001)", "custom"]
#@markdown fmax (eV/Å) used when custom is chosen.
fmax_custom = 0.02  #@param {type:"number"}
#@markdown The maximum number of steps is per structure.
max_steps = 500  #@param {type:"integer"}

#@markdown ### DFT single point (gpu4pyscf)
#@markdown Computes the DFT energies of the reported conformers. ωB97M-V/def2-TZVPD is the level of theory of UMA's training data (OMol25). def2-TZVPD can take tens of minutes per point on a T4.
run_dft = False  #@param {type:"boolean"}
dft_functional = "B3LYP-D3(BJ)"  #@param ["ωB97M-V (same as OMol25)", "B3LYP-D3(BJ)", "PBE0-D3(BJ)", "M06-2X", "r2SCAN-D4"]
dft_basis = "def2-SVP"  #@param ["def2-SVP", "def2-TZVP", "def2-TZVPD"]
#@markdown Time limit for DFT (minutes, all structures together). Past it, DFT is skipped and the UMA results are saved. 0 means no limit (DFT still stops before the Colab runtime is likely to end).
dft_time_limit = 60  #@param {type:"integer"}

In [ ]:
#@title 2. File selection and checks { display-mode: "form" }
#@markdown File selection and the token check happen here. You can leave the install and calculations after this running.

import os
import subprocess
import sys

COLABUMA_REF = "main"  # GitHub branch or tag of the shared code (colabuma package)


def _use_colabuma(ref):
    # Use colabuma next to the notebook if it is there, otherwise fetch it from GitHub
    if os.path.isdir("colabuma"):
        path = os.getcwd()
    else:
        path = "/content/ColabChem" if os.path.isdir("/content") else os.path.expanduser("~/ColabChem")
        if not os.path.isdir(os.path.join(path, "colabuma")):
            r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", ref,
                                "https://github.com/RChibaChiba/ColabChem.git", path], capture_output=True, text=True)
            if r.returncode != 0 or not os.path.isdir(os.path.join(path, "colabuma")):
                raise RuntimeError(f"Could not fetch the ColabChem shared code from GitHub (ref {ref}).\n{r.stderr}")
    if path not in sys.path:
        sys.path.insert(0, path)


_use_colabuma(COLABUMA_REF)

from colabuma import conf, dft, env, opt, output, structure, ui
from colabuma.ui import ColabUMAError, ok

ui.install_error_handler()

fmax = opt.fmax_from_form(convergence, fmax_custom)
structure.check_form(fmax, max_steps, multiplicity)
job = opt.Job(structure.out_prefix(output_name), charge, multiplicity, model_name, task_name,
              optimizer_name, fmax, max_steps, convergence)
st = conf.parse_settings(n_conformers, md_temperature, md_time_ps, n_replicas, n_snapshots, n_refine,
                         keep_coordination, random_seed)
if run_dft:
    dft.parse_method(dft_functional, dft_basis)
    if dft_time_limit < 0:
        raise ColabUMAError("The DFT time limit must be 0 or more.", "0 means no limit.",
                            ja="DFT の時間の上限は0以上にしてください。", ja_hint="0 なら上限なしです。")
env.check_gpu()
env.ensure_token()
upload_name, upload_data = structure.upload()
ok("Input checked. You can leave it running from here.")

In [ ]:
#@title 3. Setup (install and login) { display-mode: "form" }
DEVICE = env.setup(["dft"] if run_dft else [])

In [ ]:
#@title 4. Load the model { display-mode: "form" }
calc = env.load_calculator(model_name, task_name)

In [ ]:
#@title 5. Prepare the structures { display-mode: "form" }
atoms0, source_desc = structure.read_structure(upload_name, upload_data, charge, multiplicity)
ok(f"Initial structure ready: {atoms0.get_chemical_formula()} ({len(atoms0)} atoms).")
structure.show_numbered(atoms0)
if run_dft:
    dft.check_memory([("", atoms0)], charge, multiplicity, dft.parse_method(dft_functional, dft_basis)[1])

In [ ]:
#@title 6. Conformer search { display-mode: "form" }
#@markdown Optimizes the input, runs the high-temperature MD, then optimizes the snapshots in two rounds. The bars show the time left.
conf_res = conf.run(atoms0, calc, job, st)

In [ ]:
#@title 7. Results { display-mode: "form" }
files, log = conf.report(conf_res, job)

In [ ]:
#@title 8. DFT single points { display-mode: "form" }
dft_log = []
if run_dft:
    _points, _uma_rows = conf.dft_points(conf_res)
    _f, dft_log = dft.run_optional(_points, charge, multiplicity, dft_functional, dft_basis, _uma_rows, dft_time_limit)
    files.update(_f)
else:
    print("DFT single points are off.")

In [ ]:
#@title 9. Save { display-mode: "form" }
files["log.txt"] = output.log_text("ColabUMA Conf log", source_desc, job, [log, dft_log])
zip_path = output.save_zip(job.out_prefix, files)